# Step 3: Positions

<!-- notebook-only -->

> **How to use this notebook**
> - **Just reading?** The text cells contain the whole lesson. You can skip every code cell. An even nicer way to read: the website, built with `.venv/bin/python website/build.py --serve`.
> - **Want to practice?** Run the code cells from top to bottom (Shift+Enter). They recompute every table and let you experiment. In **Your turn** cells, replace `None` with your answer: `check(...)` says if it is right and gives a hint if not, and `solution("3.1")` shows the worked answer.
> - Every question also has a **Show the answer** box you can open.

<!-- /notebook-only -->

**Goal:** give each vector a sense of *where* its token sits in the text.

In [ ]:
# Run this first (practice only): imports, the tokenizer (Step 1) and the embedding table (Step 2)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from toddler.bpe import BPETokenizer
from toddler.data import CORPUS
from toddler.embeddings import COLUMNS, toddler_table
from toddler.exercises import check, solution


def short(x):
    return "0" if abs(x) < 1e-9 else f"{x:.2f}".rstrip("0").rstrip(".")


np.set_printoptions(formatter={"float_kind": short})
pd.set_option("display.float_format", short)

tok = BPETokenizer.train(CORPUS)
table = toddler_table(tok)
sentence = "I am hungry . I want milk ."
ids = tok.encode(sentence)
grid = table[ids]
print(sentence, "→", ids)

## 3.1 The problem: same token, same vector

After Step 2, a token's vector only depends on the token. Our sentence has `_I` twice, at positions 1 and 5:

| pos | token | me | be | want | stop | feel | food | hug | sleep |
|---|---|---|---|---|---|---|---|---|---|
| 1 | `_I` | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 5 | `_I` | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |

They are identical. But the first `_I` is followed by `_am`, and the second by `_want`.

A bigger problem is coming. Attention (Step 4) works by **mixing** the vectors of the text together. Mixing doesn't care about order: it only sees **which** vectors are there, like toys in a bag. Look at these two texts:

```
I want milk .    →   _I     _want  _milk  _.
milk want I .    →   _milk  _want  _I     _.
```

Both bags hold exactly the same 4 vectors: one `_I`, one `_want`, one `_milk`, one `_.`. Without positions, the model could not tell "I want milk" from "milk want I".

In [ ]:
# Try it: same token, same vector; and the same bag for two different orders
print("row 1 (_I):", grid[0])
print("row 5 (_I):", grid[4])
print("identical?", np.array_equal(grid[0], grid[4]))


def bag(text):
    """The rows of a text in sorted order: all that order-blind mixing can see."""
    return sorted(tuple(row) for row in table[tok.encode(text)])


print('\nsame bag for "I want milk ." and "milk want I ."?', bag("I want milk .") == bag("milk want I ."))

## 3.2 The idea: a vector for each position

Give every **position** its own vector, and **add** it to the token's vector:

```
input vector  =  token vector (what it is)  +  position vector (where it is)
```

Then `_I` at position 1 and `_I` at position 5 get different position vectors, so their input vectors differ.

## 3.3 The toddler's position code: a clock with one hand

Our vectors get **2 new columns** for the position, `px` and `py`. That makes **10 numbers**: 8 for meaning, 2 for position.

Picture a clock with a single hand. At position 1 it points right. Each next position turns it by **45°**, an eighth of a full turn:

- angle = (position − 1) × 45°
- `px` = how far right the hand points (the cosine of the angle)
- `py` = how far up it points (the sine of the angle)

| position | angle | px | py | the hand points |
|---|---|---|---|---|
| 1 | 0° | 1 | 0 | → |
| 2 | 45° | 0.71 | 0.71 | ↗ |
| 3 | 90° | 0 | 1 | ↑ |
| 4 | 135° | −0.71 | 0.71 | ↖ |
| 5 | 180° | −1 | 0 | ← |
| 6 | 225° | −0.71 | −0.71 | ↙ |
| 7 | 270° | 0 | −1 | ↓ |
| 8 | 315° | 0.71 | −0.71 | ↘ |

0.71 is short for 0.7071…, the cosine and sine of 45°.

![The position clock: our sentence around the clock](figures/03_position_clock.png)

The two `_I` (positions 1 and 5) now point in opposite directions.

In [ ]:
# Try it: the clock hand of each position
def position_vector(position, step_degrees=45):
    angle = np.radians((position - 1) * step_degrees)
    return np.round([np.cos(angle), np.sin(angle)], 10) + 0.0  # rounding removes float dust like 6e-17


positions = np.array([position_vector(p) for p in range(1, 9)])
pd.DataFrame(positions, index=[f"#{p}" for p in range(1, 9)], columns=["px", "py"])

## 3.4 Adding, step by step

`_I` at position 1:

```
               me   be want stop feel food  hug sleep    px    py
_I              1    0    0    0    0    0    0    0     0     0
+ position 1    0    0    0    0    0    0    0    0     1     0
= input         1    0    0    0    0    0    0    0     1     0
```

`_I` at position 5:

```
               me   be want stop feel food  hug sleep    px    py
_I              1    0    0    0    0    0    0    0     0     0
+ position 5    0    0    0    0    0    0    0    0    -1     0
= input         1    0    0    0    0    0    0    0    -1     0
```

![Adding a position to a token](figures/03_adding_positions.png)

The whole sentence, now 8 rows of 10 numbers:

| pos | token | me | be | want | stop | feel | food | hug | sleep | px | py |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 1 | `_I` | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 0 |
| 2 | `_am` | 0 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0.71 | 0.71 |
| 3 | `_hungry` | 0 | 0 | 0 | 0 | 1 | 1 | 0 | 0 | 0 | 1 |
| 4 | `_.` | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 0 | −0.71 | 0.71 |
| 5 | `_I` | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | −1 | 0 |
| 6 | `_want` | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 0 | −0.71 | −0.71 |
| 7 | `_milk` | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | −1 |
| 8 | `_.` | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 0.71 | −0.71 |

Rows 1 and 5 now differ, and so do rows 4 and 8 (the two `_.`). **This grid is what goes into the transformer.**

Why "add" when we only fill 2 empty columns? The token vector has 0 in the position columns, and the position vector has 0 in the meaning columns, so adding them just fills the empty columns. Real models don't keep separate columns: they add the position on top of the meaning in the **same** columns, and training learns to tell them apart. We keep them apart only so you can read the numbers.

In [ ]:
# Try it: add a position to a token, column by column
COLUMNS10 = COLUMNS + ["px", "py"]


def show_add(token, position):
    token_part = np.append(table[tok.token_to_id[token]], [0, 0])
    position_part = np.append(np.zeros(8), position_vector(position))
    print(f"{'':<14}" + "".join(f"{c:>6}" for c in COLUMNS10))
    for label, values in [(token, token_part), (f"+ position {position}", position_part),
                          ("= input", token_part + position_part)]:
        print(f"{label:<14}" + "".join(f"{short(x):>6}" for x in values))
    print()


show_add("_I", 1)
show_add("_I", 5)

In [ ]:
# Try it: the whole sentence with positions (8 rows of 10 numbers)
def with_positions(grid):
    n = len(grid)
    token_part = np.hstack([grid, np.zeros((n, 2))])
    position_part = np.hstack([np.zeros((n, grid.shape[1])), np.array([position_vector(p) for p in range(1, n + 1)])])
    return token_part + position_part


inputs = with_positions(grid)
pd.DataFrame(inputs, index=[f"{p} {tok.vocab[i]}" for p, i in enumerate(ids, start=1)], columns=COLUMNS10)

## 3.5 What the dot product sees now

The dot product of two position vectors only depends on **how far apart** the two positions are:

- `#1 · #2` = 1 × 0.71 + 0 × 0.71 = **0.71** (1 step apart)
- `#4 · #5` = (−0.71) × (−1) + 0.71 × 0 = **0.71** (also 1 step apart: same answer)
- `#1 · #3` = 1 × 0 + 0 × 1 = **0** (2 steps apart)
- `#1 · #5` = 1 × (−1) + 0 × 0 = **−1** (4 steps apart: opposite sides)

| distance (steps) | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| position dot product | 1 | 0.71 | 0 | −0.71 | −1 | −0.71 | 0 | 0.71 |

This is the clever part. Attention is built on dot products (Step 4), so it will be able to tell **near** tokens from **far** ones, wherever they are in the text.

Look at the end of the table: 7 steps apart gives 0.71, like 1 step apart. On the circle, position 8 sits right next to position 1. That is the limit of a single hand (section 3.6).

The whole input vectors add up both parts. For the two `_I`:

- `_I` at #1 · `_I` at #5 = 1 (same meaning) + (−1) (opposite positions) = **0**
- `_I` at #1 · `_I` at #1 = 1 + 1 = **2**

And the bag problem is solved. In "I want milk .", `_I` is at position 1, with (px, py) = (1, 0). In "milk want I .", it is at position 3, with (0, 1). The two bags now hold different vectors.

In [ ]:
# Try it: every position dot product (the same distance always gives the same number)
labels = [f"#{p}" for p in range(1, 9)]
pd.DataFrame(positions @ positions.T, index=labels, columns=labels)

In [ ]:
# Try it: the two _I, and the bag test again, now with positions
print("_I at #1 · _I at #5 =", short(inputs[0] @ inputs[4]))
print("_I at #1 · _I at #1 =", short(inputs[0] @ inputs[0]))


def bag_with_positions(text):
    return sorted(tuple(row.round(6)) for row in with_positions(table[tok.encode(text)]))


print('\nsame bag for "I want milk ." and "milk want I ."?',
      bag_with_positions("I want milk .") == bag_with_positions("milk want I ."))

## 3.6 One hand is not enough for long texts

After 8 steps the hand has made a full turn, so position 9 would point exactly like position 1. Our sentences have 8 tokens, so one hand is enough for the toddler. Real texts are much longer.

The fix is a clock with **several hands turning at different speeds**, like the second, minute and hour hands of a real clock:

| position | fast hand (45° per step) | slow hand (5.6° per step) |
|---|---|---|
| 1 | 0° → (1, 0) | 0° → (1, 0) |
| 2 | 45° → (0.71, 0.71) | 5.6° → (1.00, 0.10) |
| 9 | 360° = 0° → (1, 0) | 45° → (0.71, 0.71) |

![A fast hand and a slow hand](figures/03_two_hands.png)

- The fast hand tells **neighbors** apart, but repeats after 8 steps.
- The slow hand tells **far** positions apart, but barely moves from one step to the next.

Together they only repeat after 64 positions. Each hand needs 2 columns.

In [ ]:
# Try it: a fast hand and a slow hand
for p in [1, 2, 9]:
    print(f"#{p}:  fast hand {position_vector(p, 45)}   slow hand {position_vector(p, 45 / 8)}")

## 3.7 How real LLMs do it

- **Sine waves, like our clock** (the original Transformer, 2017): many hands, from very fast to very slow, added to the token vectors.
- **Learned positions** (GPT-2, GPT-3): a second table with one learned row per position (1,024 rows in GPT-2, 2,048 in GPT-3), added like ours. It's simple, but the model knows nothing past its last row.
- **Rotation, "RoPE"** (Llama, Mistral, Qwen and most recent models): instead of adding a position vector, attention *turns* pairs of numbers by an angle that grows with the position. These are our clock hands again, used inside attention. The dot product then depends only on the distance between two tokens.
- **Context window:** the number of positions a model can handle. Our toddler: 8. Big models today: 128,000 tokens or more.

## 3.8 What is still missing

Each row now knows **what** it is and **where** it is. But row 6 (`_want`) still knows nothing about row 3 (`_hungry`), so it can't guess `_milk`. That is the job of **Step 4, attention**: each token looks back at the earlier ones, using dot products of meaning and of position.

<!-- notebook-only -->

The same code is saved in `toddler/positions.py`, and the next notebooks import it from there.

<!-- /notebook-only -->

In [ ]:
# Try it: the package version used by the next notebooks gives the same inputs
from toddler import positions as toddler_positions

assert np.allclose(toddler_positions.with_positions(grid), inputs)
print("toddler/positions.py builds the same inputs")

## Your turn

Do each one **by hand first**, using the tables above.

### 3.1 What is the position vector (px, py) of position 7?

<details>
<summary>Show the answer</summary>
<!-- answer: 0 -1 -->
<!-- format: two numbers, e.g. 1 0 -->

Position 7 turns the hand by (7 − 1) × 45° = 270°: straight down. So (px, py) = **(0, −1)**.

</details>

In [ ]:
my_answer = None  # two numbers, e.g. [1, 0]
check("3.1", my_answer)

### 3.2 Write the full input vector (10 numbers) of `_want` at position 6.

<details>
<summary>Show the answer</summary>
<!-- answer: 0 0 1 0 0 0 0 0 -0.71 -0.71 | 0 0 1 0 0 0 0 0 -0.7 -0.7 | 0 0 1 0 0 0 0 0 -0.707 -0.707 | 0 0 1 0 0 0 0 0 -0.7071 -0.7071 -->
<!-- format: 10 numbers separated by spaces -->

```
               me   be want stop feel food  hug sleep    px    py
_want           0    0    1    0    0    0    0    0     0     0
+ position 6    0    0    0    0    0    0    0    0 -0.71 -0.71
= input         0    0    1    0    0    0    0    0 -0.71 -0.71
```

Position 6 is 225°: the hand points down and to the left.

</details>

In [ ]:
my_answer = None  # 10 numbers, e.g. [0, 0, 1, 0, 0, 0, 0, 0, 0.5, 0.5]
check("3.2", my_answer)

### 3.3 Compute the position dot product of positions 2 and 4.

<details>
<summary>Show the answer</summary>
<!-- answer: 0 -->
<!-- format: one number -->

Position 2 = (0.71, 0.71) and position 4 = (−0.71, 0.71):

0.71 × (−0.71) + 0.71 × 0.71 = −0.5 + 0.5 = **0**

They are 2 steps (90°) apart, just like positions 1 and 3: same distance, same dot product.

</details>

In [ ]:
my_answer = None  # one number
check("3.3", my_answer)

<!-- notebook-only -->

### 3.4 (code) Write `my_position_vector(position)` yourself, with `math.cos` and `math.sin`.

<details>
<summary>Show the answer</summary>

```python
import math

def my_position_vector(position):
    angle = math.radians((position - 1) * 45)
    return [math.cos(angle), math.sin(angle)]
```

</details>

<!-- /notebook-only -->

In [ ]:
import math


def my_position_vector(position):
    # your code here: the hand turns 45° per position, starting at 0° for position 1
    return None


check("3.4", my_position_vector)

### 3.5 On our one-hand clock, which position looks exactly like position 1?

<details>
<summary>Show the answer</summary>
<!-- choices: *Position 9: the hand has made a full turn | Position 5: it is on the opposite side | Position 2: it is the neighbor | None: every position always looks different -->

**Position 9**: (9 − 1) × 45° = 360°, a full turn, so the hand points exactly like at position 1. That's why real models use several hands turning at different speeds.

</details>

In [ ]:
my_answer = None  # a position number
check("3.5", my_answer)

### 3.6 Think: without positions, what would attention see in "I want milk ." and "milk want I ."?

<details>
<summary>Show the answer</summary>
<!-- choices: *The same bag of 4 vectors: it could not tell the two orders apart | Two different sentences, because the words are in a different order | Only the first token of each text | Only the last token of each text -->

The same bag of 4 vectors (`_I`, `_want`, `_milk`, `_.`), because mixing ignores order. With positions, `_I` is (1, 0) in the first text and (0, 1) in the second, so the two bags differ.

</details>

## Summary

- Same token, same vector: without help, the model can't tell *where* a token is, and attention would see the text as a bag of tokens.
- The fix: add a **position vector** to each token vector.
- The toddler's position code is a clock hand turning 45° per position, stored in 2 new columns, `px` and `py`. Vectors now have 10 numbers.
- The dot product of two position vectors depends only on their **distance**, so attention will be able to tell near from far.
- One hand repeats after 8 positions. Real models use many hands (sine waves), learned positions, or rotations (RoPE).
- Next: **Step 4, attention**.